> 在线讲义：[第 16 章 · 位置编码（下）：多档时钟](https://wikieden.github.io/transformer-visual/lessons/16-multi-clock.html) · 本书《从零读懂 Transformer》

# 第 16 章 · 位置编码（下）：多档时钟

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/16-multi-clock.html) · 仓库里 `transformer-deep-dive/notes/lessons/16-multi-clock.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 两个轮子 | 快 ¼ 圈、慢 ⅛ 圈 | `two_wheels(seat)` | 1 号 [0, 1, 0.71, 0.71]，5 号 [0, 1, −0.71, −0.71] |
| 论文的号码牌 | $PE_{(pos,2i)}$、$PE_{(pos,2i+1)}$ | `sinusoidal_pe(100, 512)` | 100 × 512 |
| 一步多长 | $1/10000^{2i/d_{model}}$ | `1 / 10000 ** (2 * i / 512)` | 第 0、128、255 个轮子：1、0.01、0.000104 |
| 近处看快轮 | 1 号 vs 2 号 | `pe[seat, 0:2]` | (0.84, 0.54) vs (0.91, −0.42) |
| 远处看慢轮 | 1 号 vs 45 号 | `pe[seat, 256:258]` | (0.01, 1.00) vs (0.43, 0.90) |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import torch

**函数 `sinusoidal_pe`**：PE[pos, 2i] = sin(pos / 10000^(2i/d))，PE[pos, 2i+1] = cos(同一个数)。返回 n_pos × d_model。

In [ ]:
def sinusoidal_pe(n_pos: int, d_model: int) -> torch.Tensor:
    """PE[pos, 2i] = sin(pos / 10000^(2i/d))，PE[pos, 2i+1] = cos(同一个数)。返回 n_pos × d_model。"""
    pos = torch.arange(n_pos, dtype=torch.float64)[:, None]  # n_pos × 1
    i = torch.arange(d_model // 2, dtype=torch.float64)[None, :]  # 1 × 256
    step = 1.0 / 10000 ** (2 * i / d_model)  # 每个轮子一步多长（弧度）
    angle = pos * step  # n_pos × 256：广播，一次算出所有座位、所有轮子
    pe = torch.zeros(n_pos, d_model, dtype=torch.float64)
    pe[:, 0::2] = torch.sin(angle)  # 偶数位放 sin
    pe[:, 1::2] = torch.cos(angle)  # 奇数位放 cos
    return pe

**函数 `two_wheels`**：第一步：快轮每座 1/4 圈、慢轮每座 1/8 圈，牌 = (快 cos, 快 sin, 慢 cos, 慢 sin)。

In [ ]:
def two_wheels(seat: int) -> torch.Tensor:
    """第一步：快轮每座 1/4 圈、慢轮每座 1/8 圈，牌 = (快 cos, 快 sin, 慢 cos, 慢 sin)。"""
    a, b = seat * 0.25 * 2 * math.pi, seat * 0.125 * 2 * math.pi
    return torch.tensor([math.cos(a), math.sin(a), math.cos(b), math.sin(b)])

**函数 `r`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def r(t: torch.Tensor, n: int = 2) -> list[float]:
    return [round(float(x), n) + 0.0 for x in t]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("两个轮子：1 号座", r(two_wheels(1)), " 5 号座", r(two_wheels(5)))
print("  快轮那两位一样，慢轮那两位不同 → 4 个数合起来不撞牌")
pe = sinusoidal_pe(100, 512)

In [ ]:
print("\n论文的号码牌: 形状", tuple(pe.shape), "（100 个座位 × 512 个数 = 256 个轮子 × (sin, cos)）")
for i in [0, 128, 255]:
    step = 1 / 10000 ** (2 * i / 512)
    print(f"  第 {i:>3} 个轮子: 一步 {step:.6f} 弧度，转一圈约 {2 * math.pi / step:,.2f} 个座位")

In [ ]:
print("\n近处看快轮、远处看慢轮（(sin, cos)）")
for seat in [1, 2, 45]:
    print(f"  {seat:>2} 号座: 第 0 个轮子 {r(pe[seat, 0:2])}   第 128 个轮子 {r(pe[seat, 256:258])}")

In [ ]:
print("\n往后挪 2 个座位，第 0 个轮子多转的角度和起点无关")
for start in [3, 10]:
    print(f"  {start} 号 → {start + 2} 号：多转 {(start + 2) - start} 弧度")
print("所有数都在 −1 到 1:", float(pe.min()) >= -1 and float(pe.max()) <= 1)

## 3. 代码里才看得到的

1. **广播一次算完所有座位、所有轮子**：`pos` 竖着（n × 1），`i` 横着（1 × 256），相除自动铺成 n × 256 的表，不用写两层循环。
2. **`0::2` 和 `1::2`**：从第 0 列开始每隔一列取（偶数列），从第 1 列开始每隔一列取（奇数列）。公式里的 $2i$、$2i+1$ 就是这两行。
3. **第 128 个轮子在第 256、257 列**：第 $i$ 个轮子占第 $2i$、$2i+1$ 两列，所以 `pe[:, 256:258]`。

## 4. 自己改一改

- 画一行 `pe[1]` 和 `pe[2]` 的前 10 个数、后 10 个数，看快轮那头差得多、慢轮那头几乎一样。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 一个轮子会撞牌，快慢轮子排一排；
> 括号里 = 座位号 × 步长：pos 管几步，i 管多长；
> 快轮管近，慢轮管远；挪几个座位，每轮转个固定角。